# End-to-End Two-Stream Fusion (live encoders, resumable checkpoints)

Trains the two encoders jointly (appearance Transformer on I3D features + skeleton GCN
on keypoints) with bidirectional cross-attention fusion and a shared CTC head. Unlike
the frozen-feature fusion study (`../02_fusion`), the CTC emission spikes align here
because the encoders are trained together.

**Durability**: every epoch writes `last.pt` atomically. If the job dies, **Run All**
resumes exactly where it stopped (`resume=True`). `best.pt` is the best-on-dev model.

Reference points: skeleton-alone ~42.6 test WER, appearance probe ~44, conf-route 42.4,
oracle 36.7, full Signformer 41.53. Goal: approach the oracle and beat every single stream.


In [1]:
# --- setup: paths, cuDNN off, keypoint layout, data + skeleton-checkpoint resolution ---
import os, sys, glob, torch
os.environ.setdefault('KP_LAYOUT', 'coco133')   # 55-joint skeleton subset
torch.backends.cudnn.enabled = False

_here = os.path.abspath('.')
CODE = None
for _k in range(6):
    _b = os.path.abspath(os.path.join(_here, *(['..'] * _k)))
    if os.path.exists(os.path.join(_b, 'csrl_skeleton', 'model.py')):
        CODE = _b; break
assert CODE, 'cannot locate code/csrl_skeleton'
E3 = os.path.join(CODE, 'distillation', 'experiments', '03_twostream_e2e')
for p in (E3, os.path.join(CODE, 'csrl_skeleton')):
    if p not in sys.path:
        sys.path.insert(0, p)
ROOT = os.path.dirname(CODE)

# 133-keypoint pickles (Phoenix-2014T.train) and raw I3D features (phoenix14t.pami0.train)
KP = None
for _c in (os.environ.get('MSKA_DATA_DIR'),
           os.path.join(ROOT, 'dataset', 'phoenix2014t_133kp'),
           os.path.join(ROOT, 'dataset', 'pose', 'phoenix2014t_133kp')):
    if _c and os.path.exists(os.path.join(_c, 'Phoenix-2014T.train')):
        KP = _c; break
assert KP, 'cannot locate phoenix2014t_133kp (Phoenix-2014T.train)'
I3D = os.path.join(ROOT, 'dataset', 'features', 'i3d_pami0')
assert os.path.exists(os.path.join(I3D, 'phoenix14t.pami0.train')), 'missing i3d_pami0'

# skeleton checkpoint for the warm start (searched in the usual places; can be overridden)
SKEL_CKPT = None
for _p in glob.glob(os.path.join(ROOT, 'dataset', 'checkpoints', '**', 'best*.pt'), recursive=True) + \
          glob.glob(os.path.join(ROOT, 'runs', '**', 'best*.pt'), recursive=True):
    if any(t in _p.lower() for t in ('skel', 'run133', 'teacher', 'pose')):
        SKEL_CKPT = _p; break
print('kp   :', KP)
print('i3d  :', I3D)
print('skel ckpt (warm-start):', SKEL_CKPT or 'NOT found -> skeleton trained from scratch (set SKEL_CKPT manually)')

kp   : /home/ebufi/Sign-Language-Recognition/dataset/phoenix2014t_133kp
i3d  : /home/ebufi/Sign-Language-Recognition/dataset/features/i3d_pami0
skel ckpt (warm-start): NON trovato -> skeleton da zero (puoi settare SKEL_CKPT a mano)


In [4]:
# --- end-to-end training, V2 (fixes BUGS.md #1 #2 #3 #6) ---
# SEPARATE run_dir (twostream_v2): the checkpoints of the (bugged) v1 run are kept
# intact in runs/twostream for the v1-vs-v2 comparison reported in the thesis.
# SKEL_CKPT must exist: if the heuristic above did not find it, set it manually, e.g.
#   SKEL_CKPT = '/home/ebufi/Sign-Language-Recognition/<path>/best.pt'

SKEL_CKPT = os.path.join(ROOT, 'runs', 'run133', 'tssi133_gcn_best.pt')
from twostream_train import train

best_dev, ckpt = train(
    run_dir=os.path.join(ROOT, 'runs', 'twostream_v2'),
    kp_dir=KP, i3d_dir=I3D, skeleton_ckpt=SKEL_CKPT,
    epochs=60, batch_size=8, lr=1e-4, weight_decay=1e-4,
    d=256, app_layers=3, heads=8, ff=1024, fusion_blocks=2, dropout=0.1,
    alpha=0.4, max_frames=400, patience=15, resume=True,
)
print('best dev WER:', round(best_dev, 2), '->', ckpt)


  merge map: 0 entries (0 hyphen-safe) | vocab: 1087 classes (<blank>=0, <unk>=1, gloss_train=1085)
  7095 joint samples | augment=True
  519 joint samples | augment=False
  642 joint samples | augment=False
[warm-start] skeleton loaded from tssi133_gcn_best.pt (missing 0, unexpected 0)
params 14.53M | classes 1087 | joints 55


/home/ebufi/venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


[e2e] ep   1 | loss 12.100 | dev 98.80% (S3 D3700 I0)  *best
[e2e] ep   2 | loss 7.497 | dev 68.92% (S658 D1915 I10)  *best
[e2e] ep   3 | loss 5.720 | dev 55.28% (S1070 D964 I38)  *best
[e2e] ep   4 | loss 4.690 | dev 48.61% (S1008 D758 I56)  *best
[e2e] ep   5 | loss 3.950 | dev 45.28% (S922 D699 I76)  *best
[e2e] ep   6 | loss 3.391 | dev 42.21% (S893 D606 I83)  *best
[e2e] ep   7 | loss 2.951 | dev 39.73% (S821 D588 I80)  *best
[e2e] ep   8 | loss 2.584 | dev 39.99% (S759 D672 I68)
[e2e] ep   9 | loss 2.288 | dev 38.74% (S817 D536 I99)  *best
[e2e] ep  10 | loss 2.049 | dev 38.82% (S728 D664 I63)
[e2e] ep  11 | loss 1.830 | dev 36.63% (S750 D527 I96)  *best
[e2e] ep  12 | loss 1.647 | dev 36.98% (S725 D561 I100)
[e2e] ep  13 | loss 1.483 | dev 36.74% (S716 D575 I86)
[e2e] ep  14 | loss 1.340 | dev 36.39% (S778 D455 I131)  *best
[e2e] ep  15 | loss 1.203 | dev 36.31% (S767 D467 I127)  *best
[e2e] ep  16 | loss 1.098 | dev 36.71% (S746 D539 I91)
[e2e] ep  17 | loss 0.984 | dev 35.38%